In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/akshaykrishnan/sipakmed5/sipakmed_5/normal_Parabasal/025_01.bmp
/kaggle/input/datasets/akshaykrishnan/sipakmed5/sipakmed_5/normal_Parabasal/001_01.bmp
/kaggle/input/datasets/akshaykrishnan/sipakmed5/sipakmed_5/normal_Parabasal/006_04.bmp
/kaggle/input/datasets/akshaykrishnan/sipakmed5/sipakmed_5/normal_Parabasal/052_04.bmp
/kaggle/input/datasets/akshaykrishnan/sipakmed5/sipakmed_5/normal_Parabasal/072_07.bmp
/kaggle/input/datasets/akshaykrishnan/sipakmed5/sipakmed_5/normal_Parabasal/051_04.bmp
/kaggle/input/datasets/akshaykrishnan/sipakmed5/sipakmed_5/normal_Parabasal/028_07.bmp
/kaggle/input/datasets/akshaykrishnan/sipakmed5/sipakmed_5/normal_Parabasal/003_13.bmp
/kaggle/input/datasets/akshaykrishnan/sipakmed5/sipakmed_5/normal_Parabasal/038_01.bmp
/kaggle/input/datasets/akshaykrishnan/sipakmed5/sipakmed_5/normal_Parabasal/047_06.bmp
/kaggle/input/datasets/akshaykrishnan/sipakmed5/sipakmed_5/normal_Parabasal/055_01.bmp
/kaggle/input/datasets/akshaykrishnan/sipak

In [2]:
import os
import copy
import random
import warnings
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

warnings.filterwarnings("ignore")
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.10.0+cu128
CUDA available: True


In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATASET_ROOT = "/kaggle/input/datasets/akshaykrishnan/sipakmed5/sipakmed_5"
OUTPUT_DIR = "/kaggle/working/efficientnet_b0_only_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASS_NAMES = [
    "abnormal_Dyskeratotic",
    "abnormal_Koilocytotic",
    "benign_Metaplastic",
    "normal_Parabasal",
    "normal_Superficial_Intermediate",
]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

IMG_SIZE = 224
BATCH_SIZE = 16
NUM_FOLDS = 5
EPOCHS = 14
BACKBONE_LR = 8e-5
HEAD_LR = 4e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
USE_AMP = torch.cuda.is_available()
PATIENCE = 4
LABEL_SMOOTHING = 0.05

print("Device:", DEVICE)
print("Output dir:", OUTPUT_DIR)

Device: cuda
Output dir: /kaggle/working/efficientnet_b0_only_results


In [4]:
def find_dataset_root():
    if os.path.isdir(DATASET_ROOT):
        return DATASET_ROOT
    for root, dirs, files in os.walk("/kaggle/input"):
        if all(cls in dirs for cls in CLASS_NAMES):
            return root
    raise FileNotFoundError("Could not find SIPaKMeD dataset root in /kaggle/input")


DATASET_ROOT = find_dataset_root()
print("Dataset root:", DATASET_ROOT)
for cls in CLASS_NAMES:
    cls_dir = os.path.join(DATASET_ROOT, cls)
    print(cls, "->", len(os.listdir(cls_dir)))

Dataset root: /kaggle/input/datasets/akshaykrishnan/sipakmed5/sipakmed_5
abnormal_Dyskeratotic -> 813
abnormal_Koilocytotic -> 825
benign_Metaplastic -> 793
normal_Parabasal -> 787
normal_Superficial_Intermediate -> 831


In [5]:
def collect_samples(root):
    valid_ext = (".bmp", ".png", ".jpg", ".jpeg", ".tif", ".tiff")
    samples = []
    for cls in CLASS_NAMES:
        cls_dir = os.path.join(root, cls)
        for fname in sorted(os.listdir(cls_dir)):
            if fname.lower().endswith(valid_ext):
                samples.append((os.path.join(cls_dir, fname), CLASS_TO_IDX[cls]))
    return samples


samples = collect_samples(DATASET_ROOT)
paths = [x[0] for x in samples]
labels = np.array([x[1] for x in samples])

print("Total images:", len(samples))
print("Class counts:")
for cls in CLASS_NAMES:
    count = sum(1 for _, y in samples if y == CLASS_TO_IDX[cls])
    print(cls, count)

Total images: 4049
Class counts:
abnormal_Dyskeratotic 813
abnormal_Koilocytotic 825
benign_Metaplastic 793
normal_Parabasal 787
normal_Superficial_Intermediate 831


In [6]:
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(12),
    transforms.ColorJitter(
        brightness=0.10, contrast=0.10, saturation=0.08, hue=0.02
    ),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])


class CytologyDataset(Dataset):
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, label = self.items[idx]
        img = Image.open(path).convert("RGB")
        img = self.transform(img)
        return img, label, path

In [7]:
class EfficientNetB0Only(nn.Module):
    """
    Pure EfficientNet-B0 baseline.

    Removed from the proposed model:
      - 1x1 channel projection
      - region-node construction
      - global graph node
      - both GAT layers
      - fuzzy confidence head
      - fuzzy training reward

    Retained:
      - ImageNet-pretrained EfficientNet-B0 feature extractor
      - global average pooling
      - standard linear classifier
    """

    def __init__(self, num_classes=5):
        super().__init__()

        backbone = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.DEFAULT
        )

        self.backbone = backbone.features
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        # Standard lightweight EfficientNet classification head
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(1280, num_classes),
        )

    def forward(self, x):
        features = self.backbone(x)
        pooled = self.global_pool(features).flatten(1)
        logits = self.classifier(pooled)
        probabilities = F.softmax(logits, dim=1)
        predictions = torch.argmax(probabilities, dim=1)

        return {
            "logits": logits,
            "probs": probabilities,
            "pred": predictions,
        }


In [8]:
def make_loaders(train_items, val_items):
    train_ds = CytologyDataset(train_items, train_tfms)
    val_ds = CytologyDataset(val_items, val_tfms)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    return train_loader, val_loader


def make_model():
    model = EfficientNetB0Only(
        num_classes=len(CLASS_NAMES)
    ).to(DEVICE)

    backbone_params = list(model.backbone.parameters())

    head_params = [
        parameter
        for name, parameter in model.named_parameters()
        if not name.startswith("backbone.")
    ]

    optimizer = torch.optim.AdamW(
        [
            {
                "params": backbone_params,
                "lr": BACKBONE_LR,
            },
            {
                "params": head_params,
                "lr": HEAD_LR,
            },
        ],
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2,
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=USE_AMP
    )

    return model, optimizer, scheduler, scaler


parameter_check_model = EfficientNetB0Only(
    num_classes=len(CLASS_NAMES)
).to(DEVICE)

TRAINABLE_PARAMETERS = sum(
    parameter.numel()
    for parameter in parameter_check_model.parameters()
    if parameter.requires_grad
)

print("EfficientNet-B0-only trainable parameters:", TRAINABLE_PARAMETERS)

del parameter_check_model

if torch.cuda.is_available():
    torch.cuda.empty_cache()


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 153MB/s]


EfficientNet-B0-only trainable parameters: 4013953


In [9]:
def train_one_epoch(model, loader, optimizer, scaler):
    model.train()

    total_loss = 0.0
    y_true = []
    y_pred = []

    for step, (images, labels, _) in enumerate(
        loader,
        start=1,
    ):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            output = model(images)

            # Pure EfficientNet baseline uses only cross-entropy.
            loss = F.cross_entropy(
                output["logits"],
                labels,
                label_smoothing=LABEL_SMOOTHING,
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)

        y_true.extend(
            labels.detach().cpu().numpy().tolist()
        )

        y_pred.extend(
            output["pred"].detach().cpu().numpy().tolist()
        )

        if step % 10 == 0 or step == len(loader):
            print(
                f"    Train iter {step}/{len(loader)} "
                f"| batch_loss={loss.item():.4f}"
            )

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )

    return (
        total_loss / len(loader.dataset),
        accuracy,
    )


@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    total_loss = 0.0
    y_true = []
    y_pred = []
    image_paths = []

    for step, (images, labels, paths_batch) in enumerate(
        loader,
        start=1,
    ):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        output = model(images)

        loss = F.cross_entropy(
            output["logits"],
            labels,
            label_smoothing=LABEL_SMOOTHING,
        )

        total_loss += loss.item() * images.size(0)

        y_true.extend(
            labels.cpu().numpy().tolist()
        )

        y_pred.extend(
            output["pred"].cpu().numpy().tolist()
        )

        image_paths.extend(
            list(paths_batch)
        )

        if step % 10 == 0 or step == len(loader):
            print(
                f"    Val   iter {step}/{len(loader)} "
                f"| batch_loss={loss.item():.4f}"
            )

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )

    precision, recall, f1_score, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        )
    )

    return (
        total_loss / len(loader.dataset),
        accuracy,
        precision,
        recall,
        f1_score,
        y_true,
        y_pred,
        image_paths,
    )


In [10]:
all_fold_metrics = []
all_reports = []
all_history = []
all_oof_predictions = []

splitter = StratifiedKFold(
    n_splits=NUM_FOLDS,
    shuffle=True,
    random_state=SEED,
)

for fold, (train_indices, validation_indices) in enumerate(
    splitter.split(paths, labels),
    start=1,
):
    print("=" * 90)
    print(f"FOLD {fold}/{NUM_FOLDS}")
    print("=" * 90)

    train_items = [
        samples[index]
        for index in train_indices
    ]

    validation_items = [
        samples[index]
        for index in validation_indices
    ]

    train_loader, validation_loader = make_loaders(
        train_items,
        validation_items,
    )

    model, optimizer, scheduler, scaler = make_model()

    best_state = None
    best_validation_accuracy = -1.0
    best_epoch = 0
    epochs_without_improvement = 0

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_accuracy = train_one_epoch(
            model,
            train_loader,
            optimizer,
            scaler,
        )

        (
            validation_loss,
            validation_accuracy,
            validation_precision,
            validation_recall,
            validation_f1,
            _,
            _,
            _,
        ) = evaluate(
            model,
            validation_loader,
        )

        scheduler.step(validation_accuracy)

        all_history.append({
            "fold": fold,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_accuracy,
            "val_loss": validation_loss,
            "val_acc": validation_accuracy,
            "val_precision_macro": validation_precision,
            "val_recall_macro": validation_recall,
            "val_f1_macro": validation_f1,
        })

        print(
            f"Epoch {epoch:02d} "
            f"| train_loss={train_loss:.4f} "
            f"| train_acc={train_accuracy:.4f} "
            f"| val_loss={validation_loss:.4f} "
            f"| val_acc={validation_accuracy:.4f} "
            f"| val_f1={validation_f1:.4f}"
        )

        if validation_accuracy > best_validation_accuracy:
            best_validation_accuracy = validation_accuracy
            best_epoch = epoch
            best_state = copy.deepcopy(
                model.state_dict()
            )
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print(
                f"Early stopping at epoch {epoch}."
            )
            break

    model.load_state_dict(best_state)

    (
        validation_loss,
        validation_accuracy,
        validation_precision,
        validation_recall,
        validation_f1,
        y_true,
        y_pred,
        validation_paths,
    ) = evaluate(
        model,
        validation_loader,
    )

    all_fold_metrics.append({
        "fold": fold,
        "best_epoch": best_epoch,
        "accuracy": validation_accuracy,
        "precision_macro": validation_precision,
        "recall_macro": validation_recall,
        "f1_macro": validation_f1,
        "n_train": len(train_items),
        "n_val": len(validation_items),
        "trainable_parameters": TRAINABLE_PARAMETERS,
    })

    report = classification_report(
        y_true,
        y_pred,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )

    all_reports.append(
        f"FOLD {fold}\n{report}\n"
    )

    print("\n" + report)

    confusion = confusion_matrix(
        y_true,
        y_pred,
    )

    pd.DataFrame(
        confusion,
        index=CLASS_NAMES,
        columns=CLASS_NAMES,
    ).to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"confusion_matrix_fold_{fold}.csv",
        )
    )

    for path, true_label, predicted_label in zip(
        validation_paths,
        y_true,
        y_pred,
    ):
        all_oof_predictions.append({
            "fold": fold,
            "image_path": path,
            "true_index": true_label,
            "predicted_index": predicted_label,
            "true_class": CLASS_NAMES[true_label],
            "predicted_class": CLASS_NAMES[predicted_label],
            "correct": true_label == predicted_label,
        })

    torch.save(
        best_state,
        os.path.join(
            OUTPUT_DIR,
            f"best_model_fold_{fold}.pt",
        ),
    )

    del model, optimizer, scheduler, scaler

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


FOLD 1/5
    Train iter 10/203 | batch_loss=1.4082
    Train iter 20/203 | batch_loss=1.4049
    Train iter 30/203 | batch_loss=1.1318
    Train iter 40/203 | batch_loss=0.9566
    Train iter 50/203 | batch_loss=0.7080
    Train iter 60/203 | batch_loss=0.6967
    Train iter 70/203 | batch_loss=0.6996
    Train iter 80/203 | batch_loss=0.5374
    Train iter 90/203 | batch_loss=0.7120
    Train iter 100/203 | batch_loss=0.4089
    Train iter 110/203 | batch_loss=0.7450
    Train iter 120/203 | batch_loss=0.4291
    Train iter 130/203 | batch_loss=0.5057
    Train iter 140/203 | batch_loss=0.5998
    Train iter 150/203 | batch_loss=0.4102
    Train iter 160/203 | batch_loss=0.8266
    Train iter 170/203 | batch_loss=0.4466
    Train iter 180/203 | batch_loss=0.5436
    Train iter 190/203 | batch_loss=0.7537
    Train iter 200/203 | batch_loss=0.4816
    Train iter 203/203 | batch_loss=0.8155
    Val   iter 10/51 | batch_loss=0.4860
    Val   iter 20/51 | batch_loss=0.4475
    Val   iter 

In [11]:
fold_df = pd.DataFrame(all_fold_metrics)
history_df = pd.DataFrame(all_history)
oof_df = pd.DataFrame(all_oof_predictions)

summary_df = pd.DataFrame([{
    "model_variant": "EfficientNet-B0 only",
    "trainable_parameters": TRAINABLE_PARAMETERS,
    "accuracy_mean": fold_df["accuracy"].mean(),
    "accuracy_std": fold_df["accuracy"].std(ddof=0),
    "precision_macro_mean": fold_df["precision_macro"].mean(),
    "recall_macro_mean": fold_df["recall_macro"].mean(),
    "f1_macro_mean": fold_df["f1_macro"].mean(),
    "oof_accuracy": oof_df["correct"].mean(),
    "oof_correct": int(oof_df["correct"].sum()),
    "oof_incorrect": int((~oof_df["correct"]).sum()),
    "total_oof_images": len(oof_df),
}])

fold_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "fold_metrics.csv",
    ),
    index=False,
)

history_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "training_history.csv",
    ),
    index=False,
)

oof_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "actual_out_of_fold_predictions.csv",
    ),
    index=False,
)

summary_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "summary_metrics.csv",
    ),
    index=False,
)

with open(
    os.path.join(
        OUTPUT_DIR,
        "classification_reports.txt",
    ),
    "w",
) as report_file:
    report_file.write(
        "\n".join(all_reports)
    )

print("\nFinal fold metrics")
print(fold_df)

print("\nSummary metrics")
print(summary_df)

print("\nSaved files in:", OUTPUT_DIR)



Final fold metrics
   fold  best_epoch  accuracy  precision_macro  recall_macro  f1_macro  \
0     1           7  0.972840         0.972782      0.972892  0.972771   
1     2           5  0.974074         0.974277      0.974262  0.974160   
2     3           5  0.977778         0.977887      0.978052  0.977737   
3     4          12  0.979012         0.979136      0.979141  0.979123   
4     5          14  0.977750         0.977950      0.977931  0.977734   

   n_train  n_val  trainable_parameters  
0     3239    810               4013953  
1     3239    810               4013953  
2     3239    810               4013953  
3     3239    810               4013953  
4     3240    809               4013953  

Summary metrics
          model_variant  trainable_parameters  accuracy_mean  accuracy_std  \
0  EfficientNet-B0 only               4013953       0.976291      0.002391   

   precision_macro_mean  recall_macro_mean  f1_macro_mean  oof_accuracy  \
0              0.976407           

In [12]:
print(
    "Ablation Architecture: Cytology Input -> "
    "ImageNet-pretrained EfficientNet-B0 -> "
    "Global Average Pooling -> "
    "Linear Classifier -> Softmax Prediction"
)

print(
    "Removed: 1x1 projection, region graph, global graph node, "
    "both GAT layers, fuzzy head, and fuzzy reward."
)


Ablation Architecture: Cytology Input -> ImageNet-pretrained EfficientNet-B0 -> Global Average Pooling -> Linear Classifier -> Softmax Prediction
Removed: 1x1 projection, region graph, global graph node, both GAT layers, fuzzy head, and fuzzy reward.
